# Interactive 3D spline skeleton for synthetic, toy, and high-dimensional datasets

This view projects the fitted skeletal splines into the first three PCA components. Local one-standard-deviation ellipses are drawn in the tangent space orthogonal to each spline and projected into the same ambient coordinates, creating a skeleton with thick bones. The selector includes the synthetic and sklearn toy datasets as well as the real high-dimensional datasets. Rotate and zoom the Plotly figure to inspect the learned backbone, ribs, and local residual thickness.

In [1]:
from pathlib import Path
import warnings
import sys

import numpy as np
from sklearn.datasets import (
    load_breast_cancer,
    load_diabetes,
    load_digits,
    load_wine,
    make_blobs,
    make_circles,
    make_classification,
    make_gaussian_quantiles,
    make_moons,
)

working_dir = Path.cwd().resolve()
notebooks_dir = working_dir / 'notebooks' if (working_dir / 'notebooks' / '__init__.py').exists() else working_dir
project_root = notebooks_dir.parent
if not (project_root / 'src' / 'skeletalembedding').exists():
    raise RuntimeError('Start Jupyter from the repository root or its notebooks/ directory')
sys.path.insert(0, str(project_root))
sys.path.insert(0, str(project_root / 'src'))
sys.path.insert(0, str(notebooks_dir))

from skeletalembedding import SkeletalEmbedding
from skeletalembedding.datasets import generate_synthetic_datasets
from skeletalembedding.visualization.interactive import plot_spline_3d

## Fit the skeletal spline network

The selector below covers all synthetic datasets, all 2D sklearn toy datasets (lifted into 3D with independent Z noise), and the real high-dimensional sklearn datasets. Labels are used only for coloring; graph fitting remains unsupervised.

In [2]:
def make_spiral(n_samples=500, noise=0.045, turns=1.15, random_state=5):
    rng = np.random.default_rng(random_state)
    theta = np.linspace(0.0, 2.0 * np.pi * turns, n_samples)
    radius = np.linspace(0.1, 1.0, n_samples)
    points = np.column_stack([radius * np.cos(theta), radius * np.sin(theta)])
    points += rng.normal(scale=noise, size=points.shape)
    return points[rng.permutation(n_samples)]

def lift_planar_dataset(dataset, z_noise=0.045, random_state=0):
    # Add independent Z noise while keeping the signal in the XY plane.
    points, labels = dataset
    points = np.asarray(points, dtype=float)
    if points.shape[1] != 2:
        raise ValueError('lift_planar_dataset expects a two-dimensional dataset')
    rng = np.random.default_rng(random_state)
    points_3d = np.column_stack([
        points,
        rng.normal(scale=z_noise, size=len(points)),
    ])
    return points_3d, np.asarray(labels)

synthetic_datasets = {
    f'synthetic/{name}': (points, np.zeros(len(points), dtype=int))
    for name, points in generate_synthetic_datasets(
        n=500, noise=0.045, random_state=0, binary_tree_depth=3,
    ).items()
}
toy_datasets_2d = {
    'toy/moons': make_moons(n_samples=500, noise=0.07, random_state=0),
    'toy/circles': make_circles(n_samples=500, factor=0.42, noise=0.045, random_state=1),
    'toy/spiral': (make_spiral(), np.zeros(500, dtype=int)),
    'toy/blobs': make_blobs(
        n_samples=500, centers=[(-1.2, -0.8), (0.0, 1.0), (1.2, -0.4)],
        cluster_std=[0.22, 0.28, 0.20], random_state=2,
    ),
    'toy/classification': make_classification(
        n_samples=500, n_features=2, n_redundant=0, n_informative=2,
        n_clusters_per_class=1, class_sep=1.25, flip_y=0.04, random_state=3,
    ),
    'toy/gaussian-quantiles': make_gaussian_quantiles(
        n_samples=500, n_features=2, n_classes=3, random_state=4,
    ),
}
synthetic_datasets_2d = {
    name: dataset for name, dataset in synthetic_datasets.items()
    if dataset[0].shape[1] == 2
}
synthetic_datasets_3d = {
    name: dataset for name, dataset in synthetic_datasets.items()
    if dataset[0].shape[1] == 3
}
planar_datasets = {**synthetic_datasets_2d, **toy_datasets_2d}
planar_datasets = {
    name: lift_planar_dataset(dataset, random_state=index)
    for index, (name, dataset) in enumerate(planar_datasets.items())
}
high_dim_datasets = {
    'high-dimensional/digits': (load_digits().data, load_digits().target),
    'high-dimensional/wine': (load_wine().data, load_wine().target),
    'high-dimensional/breast-cancer': (load_breast_cancer().data, load_breast_cancer().target),
    'high-dimensional/diabetes': (load_diabetes().data, load_diabetes().target),
}
dataset_catalog = {**planar_datasets, **synthetic_datasets_3d, **high_dim_datasets}
list(dataset_catalog)

['synthetic/line',
 'synthetic/star',
 'synthetic/circle',
 'synthetic/figure-eight',
 'synthetic/binary-tree',
 'synthetic/loop-branch',
 'synthetic/polygon-rays-circles',
 'toy/moons',
 'toy/circles',
 'toy/spiral',
 'toy/blobs',
 'toy/classification',
 'toy/gaussian-quantiles',
 'synthetic/torus',
 'high-dimensional/digits',
 'high-dimensional/wine',
 'high-dimensional/breast-cancer',
 'high-dimensional/diabetes']

## Rotate the 3D spline skeleton

Point color is the digit target. Hover a point for its skeletal-element assignment, longitudinal coordinate, residual norm, and PCA coordinates. Each spline is rendered with sampled 1σ cross-section ellipses.

In [3]:
import ipywidgets as widgets
from IPython.display import clear_output, display
style = {'description_width': 'initial'}
control_width = widgets.Layout(width='290px')

dataset_selector = widgets.Dropdown(
    options=list(dataset_catalog),
    value='synthetic/figure-eight',
    description='dataset',
    layout=widgets.Layout(width='520px'),
)
points_fraction_slider = widgets.IntSlider(
    value=100, min=10, max=100, step=10,
    description='points (%)', continuous_update=False,
    layout=widgets.Layout(width='220px'),
)
render_button = widgets.Button(
    description='Render selected dataset',
    button_style='primary',
    layout=widgets.Layout(width='260px'),
)
smoothness_slider = widgets.FloatSlider(
    value=0.02,
    min=0.0,
    max=0.25,
    step=0.005,
    description='smoothness',
    continuous_update=False,
    readout_format='.3f',
    layout=widgets.Layout(width='240px'),
)
centroid_slider = widgets.IntSlider(
    value=24, min=8, max=96, step=4,
    description='centroids', continuous_update=False,
    layout=widgets.Layout(width='220px'),
)
max_cycles_slider = widgets.IntSlider(
    value=4, min=0, max=8, step=1,
    description='max cycles', continuous_update=False,
    layout=widgets.Layout(width='220px'),
)
topology_neighbors_slider = widgets.IntSlider(
    value=6, min=2, max=50, step=1,
    description='topology neighbors', continuous_update=False,
    layout=widgets.Layout(width='220px'),
)
local_pca_neighbors_slider = widgets.IntSlider(
    value=20, min=2, max=50, step=1,
    description='local PCA neighbors', continuous_update=False,
    layout=widgets.Layout(width='220px'),
)
persistence_cap_slider = widgets.IntSlider(
    value=60, min=10, max=1000, step=10,
    description='topology subsample cap', continuous_update=False,
    layout=widgets.Layout(width='240px'),
)
z_scale_slider = widgets.FloatSlider(
    value=1.0, min=0.1, max=3.0, step=0.1, readout_format='.1f',
    description='z scale', continuous_update=False,
    layout=widgets.Layout(width='220px'),
)
point_size_slider = widgets.FloatSlider(
    value=3.5, min=1.0, max=12.0, step=0.5, readout_format='.1f',
    description='point size', continuous_update=False,
    layout=widgets.Layout(width='220px'),
)
spline_samples_slider = widgets.IntSlider(
    value=24, min=4, max=64, step=4,
    description='spline samples', continuous_update=False,
    layout=widgets.Layout(width='220px'),
)
ellipse_samples_slider = widgets.IntSlider(
    value=32, min=4, max=64, step=4,
    description='ellipse samples', continuous_update=False,
    layout=widgets.Layout(width='220px'),
)
ellipse_bandwidth_slider = widgets.FloatSlider(
    value=0.08, min=0.01, max=0.30, step=0.01, readout_format='.2f',
    description='ellipse bandwidth', continuous_update=False,
    layout=widgets.Layout(width='240px'),
)
ellipse_scale_slider = widgets.FloatSlider(
    value=1.0, min=0.0, max=3.0, step=0.1, readout_format='.1f',
    description='ellipse scale', continuous_update=False,
    layout=widgets.Layout(width='220px'),
)
junction_scale_slider = widgets.FloatSlider(
    value=1.0, min=0.0, max=3.0, step=0.1, readout_format='.1f',
    description='junction scale', continuous_update=False,
    layout=widgets.Layout(width='220px'),
)
show_observations_checkbox = widgets.Checkbox(
    value=True, description='show observations',
)
show_nodes_checkbox = widgets.Checkbox(
    value=True, description='show nodes',
)
show_graph_checkbox = widgets.Checkbox(
    value=False, description='show reduced graph',
)
model_neighbors_slider = widgets.IntSlider(
    value=6, min=2, max=50, step=1, description='model kNN neighbors',
    continuous_update=False, layout=control_width,
)
mutual_knn_checkbox = widgets.Checkbox(value=True, description='mutual kNN', layout=control_width)
add_mst_checkbox = widgets.Checkbox(value=True, description='add Euclidean MST', layout=control_width)
spline_control_selector = widgets.Dropdown(
    options=[('support points', 'support'), ('backbone anchors', 'backbone')],
    value='backbone', description='spline controls', layout=control_width,
)
routing_length_slider = widgets.FloatSlider(value=1.0, min=0.0, max=5.0, step=0.1, readout_format='.1f', description='routing length weight', continuous_update=False, layout=control_width)
routing_tangent_slider = widgets.FloatSlider(value=1.0, min=0.0, max=5.0, step=0.1, readout_format='.1f', description='routing tangent weight', continuous_update=False, layout=control_width)
routing_density_slider = widgets.FloatSlider(value=0.5, min=0.0, max=5.0, step=0.1, readout_format='.1f', description='routing density weight', continuous_update=False, layout=control_width)
threshold_mode = widgets.Dropdown(options=[('automatic', 'auto'), ('manual', 'manual')], value='auto', description='H1 threshold', layout=control_width)
threshold_slider = widgets.FloatSlider(value=0.25, min=0.0, max=1.5, step=0.025, readout_format='.3f', description='manual threshold', continuous_update=False, layout=control_width)
detect_cycles_checkbox = widgets.Checkbox(value=True, description='detect cycles', layout=control_width)
detect_junctions_checkbox = widgets.Checkbox(value=True, description='detect junctions', layout=control_width)
local_pca_checkbox = widgets.Checkbox(value=True, description='use local PCA', layout=control_width)
tangent_boundary_checkbox = widgets.Checkbox(value=True, description='tangent boundary conditions', layout=control_width)
merge_slider = widgets.FloatSlider(value=0.0, min=0.0, max=1.0, step=0.025, readout_format='.3f', description='junction merge (0=auto)', continuous_update=False, layout=control_width)
branch_angle_slider = widgets.FloatSlider(value=45.0, min=5.0, max=180.0, step=5.0, readout_format='.0f', description='maximum branch angle', continuous_update=False, layout=control_width)
electrical_metric_selector = widgets.Dropdown(
    options=[('none', 'none'), ('effective resistance', 'effective_resistance'), ('edge leverage', 'edge_leverage'), ('aggregate current', 'aggregate_current')],
    value='none', description='electrical diagnostic', layout=control_width,
)
electrical_weight_slider = widgets.FloatSlider(value=1.0, min=0.0, max=5.0, step=0.1, readout_format='.1f', description='electrical routing weight', continuous_update=False, layout=control_width)
kron_checkbox = widgets.Checkbox(value=False, description='Kron reduction', layout=control_width)
residual_dim_slider = widgets.IntSlider(value=0, min=0, max=8, step=1, description='maximum residual dimension', continuous_update=False, layout=control_width)
residual_bandwidth_slider = widgets.FloatSlider(value=0.1, min=0.01, max=1.0, step=0.01, readout_format='.2f', description='residual PCA bandwidth', continuous_update=False, layout=control_width)
residual_smoothness_slider = widgets.FloatSlider(value=0.0, min=0.0, max=1.0, step=0.05, readout_format='.2f', description='residual basis smoothness', continuous_update=False, layout=control_width)
coverage_checkbox = widgets.Checkbox(value=False, description='coverage refinement', layout=control_width)
coverage_mode = widgets.Dropdown(options=[('automatic tolerance', 'auto'), ('absolute tolerance', 'absolute'), ('relative tolerance', 'relative')], value='auto', description='coverage tolerance', layout=control_width)
coverage_tolerance_slider = widgets.FloatSlider(value=0.25, min=0.0, max=2.0, step=0.025, readout_format='.3f', description='coverage tolerance value', continuous_update=False, layout=control_width)
coverage_quantile_slider = widgets.FloatSlider(value=0.95, min=0.5, max=1.0, step=0.01, readout_format='.2f', description='coverage quantile', continuous_update=False, layout=control_width)
coverage_iterations_slider = widgets.IntSlider(value=10, min=1, max=20, step=1, description='coverage iterations', continuous_update=False, layout=control_width)
coverage_ribs_slider = widgets.IntSlider(value=0, min=0, max=30, step=1, description='maximum ribs (0=unlimited)', continuous_update=False, layout=control_width)
coverage_selection_selector = widgets.Dropdown(options=[('greedy', 'greedy'), ('MIP', 'mip')], value='greedy', description='rib selection', layout=control_width)
rib_candidate_type_selector = widgets.Dropdown(options=[('transverse', 'transverse'), ('parallel', 'parallel'), ('both', 'both')], value='transverse', description='rib candidate type', layout=control_width)
coverage_gain_slider = widgets.FloatSlider(value=0.0, min=0.0, max=1.0, step=0.01, readout_format='.2f', description='minimum rib gain', continuous_update=False, layout=control_width)
coverage_candidates_slider = widgets.IntSlider(value=20, min=1, max=50, step=1, description='rib candidates / iteration', continuous_update=False, layout=control_width)
coverage_spacing_slider = widgets.FloatSlider(value=0.0, min=0.0, max=2.0, step=0.025, readout_format='.3f', description='candidate spacing (0=auto)', continuous_update=False, layout=control_width)
coverage_min_error_slider = widgets.FloatSlider(value=0.0, min=0.0, max=2.0, step=0.025, readout_format='.3f', description='minimum seed error (0=auto)', continuous_update=False, layout=control_width)
coverage_length_penalty_slider = widgets.FloatSlider(value=0.0, min=0.0, max=2.0, step=0.05, readout_format='.2f', description='rib length penalty', continuous_update=False, layout=control_width)
coverage_rib_penalty_slider = widgets.FloatSlider(value=0.0, min=0.0, max=2.0, step=0.05, readout_format='.2f', description='rib count penalty', continuous_update=False, layout=control_width)
coverage_junction_penalty_slider = widgets.FloatSlider(value=0.0, min=0.0, max=2.0, step=0.05, readout_format='.2f', description='junction penalty', continuous_update=False, layout=control_width)
stability_checkbox = widgets.Checkbox(value=False, description='stability selection', layout=control_width)
stability_runs_slider = widgets.IntSlider(value=5, min=1, max=30, step=1, description='subsample runs', continuous_update=False, layout=control_width)
stability_fraction_slider = widgets.FloatSlider(value=0.7, min=0.1, max=1.0, step=0.05, readout_format='.2f', description='subsample fraction', continuous_update=False, layout=control_width)
stability_support_slider = widgets.FloatSlider(value=0.75, min=0.0, max=1.0, step=0.05, readout_format='.2f', description='minimum stability support', continuous_update=False, layout=control_width)
stability_jitter_slider = widgets.FloatSlider(value=0.0, min=0.0, max=1.0, step=0.05, readout_format='.2f', description='stability jitter', continuous_update=False, layout=control_width)
stability_residual_checkbox = widgets.Checkbox(value=False, description='stable residual subspaces', layout=control_width)
rib_stability_runs_slider = widgets.IntSlider(value=0, min=0, max=30, step=1, description='rib stability runs (0=off)', continuous_update=False, layout=control_width)
rib_min_support_slider = widgets.FloatSlider(value=0.6, min=0.0, max=1.0, step=0.05, readout_format='.2f', description='minimum rib support', continuous_update=False, layout=control_width)
render_output = widgets.Output()

def fit_selected_dataset(name):
    global X, y, model, result, figure
    base_X, base_y = dataset_catalog[name]
    point_fraction = int(points_fraction_slider.value) / 100.0
    n_points = max(1, round(len(base_X) * point_fraction))
    indices = np.linspace(0, len(base_X) - 1, n_points, dtype=int)
    X, y = base_X[indices], base_y[indices]
    is_binary_tree = name == 'synthetic/binary-tree'
    is_planar = name in planar_datasets
    smoothness = float(smoothness_slider.value)
    threshold = 4.0 if threshold_mode.value == 'auto' and is_planar else (threshold_slider.value if threshold_mode.value == 'manual' else None)
    merge_distance = None if merge_slider.value == 0 else merge_slider.value
    electrical_metric = electrical_metric_selector.value
    coverage_tolerance = coverage_tolerance_slider.value if coverage_mode.value == 'absolute' else None
    coverage_relative_tolerance = coverage_tolerance_slider.value if coverage_mode.value == 'relative' else None
    coverage_max_ribs = None if coverage_ribs_slider.value == 0 else coverage_ribs_slider.value
    model = SkeletalEmbedding(
        n_centroids=min(centroid_slider.value, len(X)),
        n_neighbors=min(model_neighbors_slider.value, max(2, len(X) - 1)),
        topology_neighbors=min(topology_neighbors_slider.value, max(2, len(X) - 1)),
        mutual_knn=mutual_knn_checkbox.value,
        add_mst=add_mst_checkbox.value,
        spline_control_mode=spline_control_selector.value,
        persistence_threshold=threshold,
        spline_smoothing=smoothness,
        routing_length_weight=routing_length_slider.value,
        routing_tangent_weight=routing_tangent_slider.value,
        routing_density_weight=routing_density_slider.value,
        max_cycles=0 if is_binary_tree else max_cycles_slider.value,
        random_state=0,
        # Preserve the XY metric for lifted planar data: its small Z noise
        # is measurement thickness, not a reason to rescale the topology.
        standardize=(not is_planar if standardize_selector.value == 'auto' else standardize_selector.value == 'true'),
        persistence_max_points=min(persistence_cap_slider.value, len(X)),
        spline_samples_per_node=12,
        detect_cycles=detect_cycles_checkbox.value,
        detect_junctions=detect_junctions_checkbox.value,
        use_local_pca=local_pca_checkbox.value,
        local_pca_neighbors=min(local_pca_neighbors_slider.value, max(2, len(X) - 1)),
        max_branch_angle_degrees=branch_angle_slider.value,
        use_tangent_boundary_conditions=tangent_boundary_checkbox.value,
        merge_junction_distance=merge_distance,
        use_effective_resistance=electrical_metric in {'effective_resistance', 'edge_leverage'},
        use_electrical_flow=electrical_metric == 'aggregate_current',
        use_kron_reduction=kron_checkbox.value,
        routing_resistance_weight=electrical_weight_slider.value if electrical_metric in {'effective_resistance', 'edge_leverage'} else 0.0,
        routing_current_weight=electrical_weight_slider.value if electrical_metric == 'aggregate_current' else 0.0,
        max_residual_dim=min(residual_dim_slider.value, max(0, X.shape[1] - 1)),
        residual_pca_bandwidth=residual_bandwidth_slider.value,
        residual_subspace_smoothness=residual_smoothness_slider.value,
        coverage_refinement=coverage_checkbox.value,
        coverage_error_tolerance=coverage_tolerance,
        coverage_relative_tolerance=coverage_relative_tolerance,
        coverage_quantile=coverage_quantile_slider.value,
        coverage_max_iterations=coverage_iterations_slider.value,
        coverage_max_ribs=coverage_max_ribs,
        coverage_selection=coverage_selection_selector.value,
        rib_candidate_type=rib_candidate_type_selector.value,
        coverage_min_gain=coverage_gain_slider.value,
        coverage_max_candidates_per_iteration=coverage_candidates_slider.value,
        coverage_candidate_spacing=None if coverage_spacing_slider.value == 0 else coverage_spacing_slider.value,
        coverage_min_error=None if coverage_min_error_slider.value == 0 else coverage_min_error_slider.value,
        coverage_length_penalty=coverage_length_penalty_slider.value,
        coverage_rib_penalty=coverage_rib_penalty_slider.value,
        coverage_junction_penalty=coverage_junction_penalty_slider.value,
        stability_selection=stability_checkbox.value,
        stability_runs=stability_runs_slider.value,
        stability_fraction=stability_fraction_slider.value,
        stability_min_support=stability_support_slider.value,
        stability_jitter=stability_jitter_slider.value,
        stability_residual_subspaces=stability_residual_checkbox.value,
        rib_stability_runs=None if rib_stability_runs_slider.value == 0 else rib_stability_runs_slider.value,
        rib_min_support=rib_min_support_slider.value,
    )
    with warnings.catch_warnings():
        warnings.filterwarnings(
            'ignore',
            message='Topological landmark constraints could not all be realized by the routing substrate\\.',
            category=RuntimeWarning,
        )
        result = model.fit_transform(X)
    print(
        f'dataset: {name} | observations: {len(X)} | features: {X.shape[1]}\n'
        f'cycles: {model.realized_cycle_count_} | junctions: {len(model.junctions_)} | '
        f'smoothness: {smoothness:.3f} | '
        f'splines: {len(model.splines_)} | ribs: {len(model.rib_paths_)} | '
        f'median residual: {np.median(result.residual_norm):.4f}'
    )
    # Avoid one Plotly trace per unique value for continuous targets.
    unique_labels = np.unique(y)
    plot_labels = y if 1 < len(unique_labels) <= 24 else None
    figure = plot_spline_3d(
        model,
        result,
        labels=plot_labels,
        title=f'{name}: spline skeleton with 1σ tangent-space sections',
        z_scale=z_scale_slider.value,
        point_size=point_size_slider.value,
        show_nodes=show_nodes_checkbox.value,
        n_spline_samples=spline_samples_slider.value,
        ellipse_samples=ellipse_samples_slider.value,
        ellipse_bandwidth=ellipse_bandwidth_slider.value,
        ellipse_scale=ellipse_scale_slider.value,
        junction_ellipsoid_scale=junction_scale_slider.value,
        show_observations=show_observations_checkbox.value,
        show_reduced_graph=show_graph_checkbox.value,
    )
    figure.show()

def render_selected_dataset(_=None):
    with render_output:
        clear_output(wait=True)
        fit_selected_dataset(dataset_selector.value)

standardize_selector = widgets.Dropdown(
    options=[('automatic', 'auto'), ('always standardize', 'true'), ('preserve input scale', 'false')],
    value='auto', description='feature scaling', layout=control_width,
)
def section(title, *children):
    row_layout = widgets.Layout(display='flex', flex_flow='row wrap', align_items='center', gap='10px')
    return widgets.VBox([
        widgets.HTML(value=f'<b>{title}</b>'),
        widgets.HBox(list(children), layout=row_layout),
    ], layout=widgets.Layout(border='1px solid #dddddd', padding='7px 10px', margin='3px 0', width='100%'))
all_controls = [
    dataset_selector, points_fraction_slider, smoothness_slider, centroid_slider,
    model_neighbors_slider, topology_neighbors_slider,
    mutual_knn_checkbox, add_mst_checkbox, spline_control_selector,
    routing_length_slider, routing_tangent_slider, routing_density_slider,
    max_cycles_slider, threshold_mode, threshold_slider, persistence_cap_slider,
    detect_cycles_checkbox, detect_junctions_checkbox, tangent_boundary_checkbox,
    merge_slider, local_pca_checkbox, local_pca_neighbors_slider, branch_angle_slider,
    standardize_selector, electrical_metric_selector, electrical_weight_slider, kron_checkbox,
    residual_dim_slider, residual_bandwidth_slider, residual_smoothness_slider,
    coverage_checkbox, coverage_mode, coverage_tolerance_slider, coverage_quantile_slider,
    coverage_iterations_slider, coverage_ribs_slider, coverage_selection_selector,
    rib_candidate_type_selector, coverage_gain_slider, coverage_candidates_slider,
    coverage_spacing_slider, coverage_min_error_slider, coverage_length_penalty_slider,
    coverage_rib_penalty_slider, coverage_junction_penalty_slider, stability_checkbox,
    stability_runs_slider, stability_fraction_slider, stability_support_slider,
    stability_jitter_slider, stability_residual_checkbox, rib_stability_runs_slider,
    rib_min_support_slider, z_scale_slider, point_size_slider, spline_samples_slider,
    ellipse_samples_slider, ellipse_bandwidth_slider, ellipse_scale_slider,
    junction_scale_slider, show_observations_checkbox, show_nodes_checkbox, show_graph_checkbox,
]
for control in all_controls: control.style = style
coverage_mode.observe(lambda change: setattr(coverage_tolerance_slider, 'disabled', change['new'] == 'auto'), names='value')
display(widgets.VBox([
    section('Data', dataset_selector, points_fraction_slider),
    section('Graph fitting', centroid_slider, model_neighbors_slider, topology_neighbors_slider, mutual_knn_checkbox, add_mst_checkbox, smoothness_slider, spline_control_selector, routing_length_slider, routing_tangent_slider, routing_density_slider, standardize_selector),
    section('Topology', max_cycles_slider, threshold_mode, threshold_slider, persistence_cap_slider, detect_cycles_checkbox, detect_junctions_checkbox, tangent_boundary_checkbox, merge_slider, local_pca_checkbox, local_pca_neighbors_slider, branch_angle_slider),
    section('Electrical diagnostics', electrical_metric_selector, electrical_weight_slider, kron_checkbox),
    section('Residual and coverage', residual_dim_slider, residual_bandwidth_slider, residual_smoothness_slider, coverage_checkbox, coverage_mode, coverage_tolerance_slider, coverage_quantile_slider, coverage_iterations_slider, coverage_ribs_slider, coverage_selection_selector, rib_candidate_type_selector, coverage_gain_slider, coverage_candidates_slider, coverage_spacing_slider, coverage_min_error_slider, coverage_length_penalty_slider, coverage_rib_penalty_slider, coverage_junction_penalty_slider),
    section('Stability and subsampling', stability_checkbox, stability_runs_slider, stability_fraction_slider, stability_support_slider, stability_jitter_slider, stability_residual_checkbox, rib_stability_runs_slider, rib_min_support_slider),
    section('3D display', z_scale_slider, point_size_slider, spline_samples_slider, ellipse_samples_slider, ellipse_bandwidth_slider, ellipse_scale_slider, junction_scale_slider, show_observations_checkbox, show_nodes_checkbox, show_graph_checkbox),
    render_button, render_output,
]))
for control in all_controls: control.observe(render_selected_dataset, names='value')
render_selected_dataset()